# 1. Download Solar.ChemDX data

Download the group archives available to your account, choose how to save the
JSON records, and select which measurements to keep before saving.
Each run creates a new folder.

**Before starting:** keep all files in this folder together (including
`solarcell_tools.py`), install `requirements-notebooks.txt`, and open JupyterLab from
this folder. Use Python 3.8+. Generate an API key at
[Solar.ChemDX → Account](https://solar.chemdx.org/account).

Run the cells from top to bottom. You can enter credentials when prompted or set
`SOLARCELL_EMAIL` and `SOLARCELL_API_KEY` in your environment before starting Jupyter.
If your organisation uses a TLS proxy, set `REQUESTS_CA_BUNDLE` to its CA file.

## Save mode

Set `SAVE_MODE` in the next cell:

| Value       | Saved output |
| ---         | --- |
| `"both"`    | Save the selected data as group-specific JSON files and merged records.json |
| `"grouped"` | Save only the individual JSON files in each group folder; do not create a merged file |
| `"merged"`  | Save only the merged records.json file |

`both` and `grouped` retain original ZIPs only when `MEASUREMENT = "ALL"`.
With a specific measurement selection, original ZIPs and extracted files are
temporary in every mode; the saved group JSON also contains only the selection.
In `merged` mode, the run folder contains just `records.json`.
A completed-run pointer, `<DOWNLOAD_ROOT>/latest.json`, is maintained in every mode.

## Group and measurement selection

Use `GROUP_IDS = "ALL"` for all groups, or a list such as `[2, 3]`.
Set `MEASUREMENT` to one key or several keys before running the download cell:

```python
MEASUREMENT = "ALL"                # All devices and all measurements
MEASUREMENT = "JV"                 # Only JV
MEASUREMENT = ["JV", "PL", "SEM"]  # Any of these measurements
```

Use one of the assignments above. A device is included if it has at least one of
the requested measurements. Only the selected keys remain in `analysis` and
`analysisInfo`; legacy top-level `JV` is kept only when JV is selected. Device IDs,
group information, recipe/input data and other device metadata are preserved.
Keys are case-insensitive and matched exactly: **PL does not include TRPL**.
Comma-separated strings such as `"JV, PL"` and tuples such as `("JV", "PL")`
are also accepted. `"ALL"` must be used alone; legacy `None` still means all.

The selection is applied **before saving**. For `SAVE_MODE = "merged"`, only the
selected data goes into `records.json`; no `filtered_records.json` is created.
An unmatched selection saves an empty result. The API still downloads each
selected group's ZIP, then applies this selection locally.


In [ ]:
import os
from getpass import getpass
from pathlib import Path
from collections import Counter

from solarcell_tools import download_dataset, load_records, measurement_types

DOWNLOAD_ROOT = Path("downloads2")
GROUP_IDS = "ALL"  # All available groups, or a list such as [2, 3]. IDs are checked against the API.
SAVE_MODE = "merged"  # "grouped": group-specific JSON  / "merged": merged JSON / "both": both
MEASUREMENT = ["JV", "PL"]  # "ALL", one key such as "JV", or a list such as ["JV", "PL", "SEM"].


## Authenticate and download

The API returns temporary ZIP URLs, normally valid for about five minutes. A
download receiving HTTP 403 refreshes its link once. Network timeouts, malformed
responses and authentication failures stop the run with a readable error.

No API key or signed URL is written to the output files. The `latest.json` pointer
is updated only after every selected group has downloaded, passed record validation,
and the requested measurements have been saved in the selected format. Previous runs remain available if a new
run fails. `records_source` is a JSON file for `both`/`merged`, or the run folder
for `grouped`; both work with the inspection and CSV steps.


In [ ]:
email = os.environ.get("SOLARCELL_EMAIL") or input("Solar.ChemDX account email: ").strip()
api_key = os.environ.get("SOLARCELL_API_KEY") or getpass("Solar.ChemDX API key: ")
try:
    records_source = download_dataset(
        email, api_key, DOWNLOAD_ROOT, GROUP_IDS,
        save_mode=SAVE_MODE, measurement=MEASUREMENT
    )
finally:
    del api_key


## Inspect the saved selection

The data below has already been filtered and saved. Counts describe the devices
and measurement keys retained by your selection. `analysis` and `analysisInfo`
are checked for non-empty entries; metadata may indicate an attachment even when
parsed measurement values are absent. This notebook does not separately fetch
original images or other measurement attachments.

Changing `MEASUREMENT` requires rerunning the download cell to create a new run.
This inspection cell only reads the saved result and does not create another
JSON file. `latest.json` points to this selected result for CSV conversion.


In [ ]:
records = load_records(records_source)
counts = Counter(kind for record in records for kind in measurement_types(record))
print(f"Saved devices: {len(records)}")
print("Saved measurement types:", dict(sorted(counts.items())))
print(f"CSV input: {records_source}")


## Next step

Open [02_data_csv.ipynb](02_data_csv.ipynb) and use the same `DOWNLOAD_ROOT`.
Its default input is the latest successful saved selection, including grouped
output. Set an explicit JSON file or run/group folder path for an older snapshot.
